# 16. Komparatív Többterületes Elemzés

**Cél:** a vizsgált terület (fő mintaterület) összehasonlítása a regisztrált kontroll- és benchmark-területekkel a közös, kanonikus mutatók mentén (kínálati méret, GIS-pontosság, medián fajlagos ár, vasúti immissziós hatás, állomáselérés).

---

### 📖 Módszertani megjegyzés
A területek összehasonlítása csak azonos definíciójú mutatók mentén értelmezhető — ehhez minden terület adata ugyanazon a kanonikus sémán és számítási láncon megy keresztül. A devizaeltérések miatt a fajlagos árak külön is értelmezendők.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

# Területek konfigurációjának ellenőrzése
cfg = load_areas_config()
available_areas = list_available_areas()
print(f"Regisztrált és elérhető területek: {available_areas}")


### 1. Mintaterületek Leíró Összehasonlítása és Piaci Mérete

In [ ]:
# Területek betöltése és egyesítése komparatív elemzéshez
df_comp = load_all_areas_comparison(available_areas)
elado = df_comp[df_comp['listing_type'] == 'elado'].copy()

summary = elado.groupby('area_name').agg(
    lakasok_szama=('listing_id', 'count'),
    pontos_lakasok=('minta_garantalt_pontos', lambda x: int((x == 1).sum())),
    median_nm_ar=('nm_ar_huf', 'median'),
    atlag_ar_mft=('ar_millio_ft', 'mean'),
    atlag_terulet=('alapterulet_nm', 'mean')
).reset_index()

kpis = []
for _, r in summary.iterrows():
    kpis.append((f"{r['area_name']} Kínálat", f"{r['lakasok_szama']} db", f"Garantált pontos GIS: {r['pontos_lakasok']} db", "#1e3a8a"))
    kpis.append((f"{r['area_name']} Medián Ár", fmt_huf(r['median_nm_ar']), f"Átlagár: {r['atlag_ar_mft']:.1f} M Ft", "#059669"))

display(HTML(kpi_grid_html(kpis)))
display(summary.style.format({
    'lakasok_szama': '{:,} db',
    'pontos_lakasok': '{:,} db',
    'median_nm_ar': lambda v: fmt_huf(v),
    'atlag_ar_mft': '{:.1f} M Ft',
    'atlag_terulet': '{:.1f} m²'
}))


### 2. A Vasúti Immissziós Diszkont Összehasonlítása (0-150-300-500-1000-2000 m)
A legfontosabb ökonometriai kérdés: a vágányok közvetlen szomszédságában (<150m) fellépő árhátrány mekkora a kontroll területeken?


In [ ]:
# Immissziós sávok összehasonlítása — ELEMSZÁM-KÜSZÖBBEL (N>=5) és 95% CI-vel
immisszio_stats = elado[elado['vasut_zona'].notna()].groupby(['area_name', 'vasut_zona'], observed=False).agg(
    median_nm_ar=('nm_ar_huf', 'median'),
    darab=('listing_id', 'count'),
    se_nm_ar=('nm_ar_huf', lambda x: np.std(x) / np.sqrt(len(x)))
).reset_index()
immisszio_stats = immisszio_stats[immisszio_stats['darab'] >= 5].copy()
immisszio_stats['ci_lo'] = immisszio_stats['median_nm_ar'] - 1.96 * immisszio_stats['se_nm_ar']
immisszio_stats['ci_hi'] = immisszio_stats['median_nm_ar'] + 1.96 * immisszio_stats['se_nm_ar']

fig_imm = go.Figure()
colors = {'kobanya': '#2563eb', 'wien_nordbahnhof': '#059669', 'kontroll_1': '#d97706', 'kontroll_2': '#dc2626'}
for area, grp in immisszio_stats.groupby('area_name'):
    fig_imm.add_trace(go.Bar(
        x=grp['vasut_zona'].astype(str), y=grp['median_nm_ar'],
        name=area,
        marker_color=colors.get(area, '#64748b'),
        error_y=dict(type='data', symmetric=False,
                     array=grp['ci_hi'] - grp['median_nm_ar'],
                     arrayminus=grp['median_nm_ar'] - grp['ci_lo']),
        text=[f"n={n}" for n in grp['darab']], textposition='outside'
    ))
fig_imm.update_layout(
    barmode='group',
    title='Medián négyzetméterár az immissziós sávokban (95% CI, csak N>=5 cellák)',
    xaxis_title='Immissziós sáv', yaxis_title='Medián ár/m² (Ft)',
    template=PLOTLY_TEMPLATE, height=520,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig_imm.show()
print("Az egyes sávok árszintjei NEM ceteris paribus összehasonlíthatók (eltérő összetétel) — "
      "a kontrollált összehasonlítást az alábbi pooled regresszió adja.")


In [ ]:
# Pooled hedonikus regresszió terület-fix hatásokkal és terület×immissziós sáv interakciókkal
# Deviza-normalizálás: EUR (a bécsi nm_ar_eur közvetlenül; a forintos területek 400 Ft/EUR
# feltevéssel — explicit, nem piaci árfolyam-becslés).
import statsmodels.api as sm
comp = elado.copy()
comp['nm_ar_eur'] = np.where(comp['area_name'] == 'wien_nordbahnhof',
                            comp['nm_ar_eur'], comp['nm_ar_huf'] / 400.0)
comp['log_nm_ar_eur'] = np.log(comp['nm_ar_eur'])

mp = comp[comp['minta_garantalt_pontos'] == 1].dropna(subset=[
    'log_nm_ar_eur', 'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel',
    'has_lift', 'has_erkely', 'allapot_kod', 'emelet_szam', 'epulet_kora_ev',
    'tavolsag_metro_halozati_m', 'tavolsag_vasut_halozati_m', 'vasut_zona']).copy()
mp['emelet_szam'] = mp['emelet_szam'].fillna(mp['emelet_szam'].median())
mp['epulet_kora_ev'] = mp['epulet_kora_ev'].fillna(mp['epulet_kora_ev'].median())
kontrollok = CANON_PHYSICAL
_rail_p = canonical_rail_dummies(mp)

# Terület-dummyk (referencia: kobanya) + zóna-dummyk (referencia: >2000 m)
# ROBUSZTUS zóna-dummyk: üres kategóriák eldobása, adaptív referencia (a pooled mintán)
# Az area_name a TELJES név — az interakciókhoz az azonosító (id) kell!
_name_to_id = {v['name']: k for k, v in load_areas_config()['areas'].items()}
mp['area_id'] = mp['area_name'].map(_name_to_id)
_z_p, _z_ref_p = build_immission_dummies(mp)
Xp = pd.concat([mp[kontrollok], _rail_p,
               pd.get_dummies(mp['area_id'], drop_first=True),
               _z_p], axis=1).astype(float)

# Interakciók: csak azok a terület×zóna cellák, ahol N>=10
inter_rows = []
for area in mp['area_id'].unique():
    if area == 'kobanya':
        continue  # a fő hatás már a zóna-dummykban van
    for z in [z for z in ['<150 m', '150-300 m', '300-500 m', '500-1000 m'] if z in mp['vasut_zona'].dropna().unique()]:
        n_cell = int(((mp['area_id'] == area) & (mp['vasut_zona'] == z)).sum())
        if n_cell >= 10:
            mp[f'ia_{area}__{z}'] = ((mp['area_id'] == area) & (mp['vasut_zona'] == z)).astype(int)
            Xp[f'ia_{area}__{z}'] = mp[f'ia_{area}__{z}']
            inter_rows.append((area, z, n_cell))

Xp = sm.add_constant(Xp)
Xp = drop_constant_columns(Xp)
m_pool = sm.OLS(mp['log_nm_ar_eur'], Xp).fit(cov_type='HC1')
print(f"Pooled modell: N={int(m_pool.nobs)}, R²={m_pool.rsquared:.3f} (HC1 SE-k).")
print("Terület×immissziós sáv interakciók (a bécsi diszkont a fő mintaterülethez képest, N>=10 cellák):")
for area, z, n_cell in inter_rows:
    col = f'ia_{area}__{z}'
    b = m_pool.params[col]; p = m_pool.pvalues[col]
    sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
    print(f"  {area} × {z} (N={n_cell}): {(np.exp(b)-1)*100:+.1f}% (p={p:.3f}) {sig}")

# Fő zónahatások (fő mintaterületi referencia)
print("Fő immissziós zónahatások (referencia: >2000 m):")
for col in m_pool.params.index:
    if col in _z_p.columns:
        p = m_pool.pvalues[col]
        print(f"  {col}: {(np.exp(m_pool.params[col])-1)*100:+.1f}% (p={p:.3f})")

# Interakciós együtthatók forest plotja
if inter_rows:
    rows = []
    for area, z, _ in inter_rows:
        col = f'ia_{area}__{z}'
        ci = m_pool.conf_int().loc[col]
        rows.append({'Interakció': f'{area} × {z}',
                     'Hatás (%)': (np.exp(m_pool.params[col]) - 1) * 100,
                     'CI_lo': (np.exp(ci[0]) - 1) * 100, 'CI_hi': (np.exp(ci[1]) - 1) * 100})
    rdf = pd.DataFrame(rows)
    fig_pool = go.Figure()
    fig_pool.add_trace(go.Scatter(x=rdf['Hatás (%)'], y=rdf['Interakció'], mode='markers',
                                 error_x=dict(type='data', symmetric=False,
                                              array=rdf['CI_hi'] - rdf['Hatás (%)'],
                                              arrayminus=rdf['Hatás (%)'] - rdf['CI_lo']),
                                 marker=dict(size=11, color='#7c3aed')))
    fig_pool.add_vline(x=0, line_dash='dash', line_color='black')
    fig_pool.update_layout(title='Terület×immissziós sáv interakciók (95% CI) — a diszkont területi heterogenitása',
                           xaxis_title='Interakciós hatás (%)', template=PLOTLY_TEMPLATE, height=420)
    fig_pool.show()


### 3. Állomáskörnyéki Gyalogos Elérhetőség (TOD Prémium) az Izokrón Sávokban (375m, 750m, 1125m)

In [ ]:
# Izokrón összehasonlítás — elemszám-küszöb (N>=5) + 95% CI
izokron_lista = []
for p, m_dist in [(5, '0-375 m (5 perc)'), (10, '375-750 m (10 perc)'), (15, '750-1125 m (15 perc)')]:
    col = f'vasut_{p}p_seta'
    if col in elado.columns:
        for area_name, grp in elado.groupby('area_name'):
            minta = grp[grp[col] == 1]
            if len(minta) >= 5:
                izokron_lista.append({
                    'Terület': area_name,
                    'Gyalogos Izokrón': m_dist,
                    'Lakásszám (db)': len(minta),
                    'Medián Ár/m²': minta['nm_ar_huf'].median(),
                    'CI_lo': minta['nm_ar_huf'].median() - 1.96 * minta['nm_ar_huf'].std() / np.sqrt(len(minta)),
                    'CI_hi': minta['nm_ar_huf'].median() + 1.96 * minta['nm_ar_huf'].std() / np.sqrt(len(minta)),
                })

df_izo_comp = pd.DataFrame(izokron_lista)

fig_izo = go.Figure()
for area, grp in df_izo_comp.groupby('Terület'):
    fig_izo.add_trace(go.Bar(
        x=grp['Gyalogos Izokrón'], y=grp['Medián Ár/m²'], name=area,
        error_y=dict(type='data', symmetric=False,
                     array=grp['CI_hi'] - grp['Medián Ár/m²'],
                     arrayminus=grp['Medián Ár/m²'] - grp['CI_lo']),
        text=[f"n={n}" for n in grp['Lakásszám (db)']], textposition='outside'
    ))
fig_izo.update_layout(
    barmode='group',
    title='Állomáskörnyéki TOD elérhetőség (0-375/375-750/750-1125 m, hálózati) — 95% CI, N>=5',
    xaxis_title='Hálózati gyalogos izokrón (1.25 m/s)', yaxis_title='Medián ár/m² (Ft)',
    template=PLOTLY_TEMPLATE, height=480,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig_izo.show()


### 💡 4. Tudományos Szintézis és TDK Eredmények
- **Robusztusság igazolása**: A kontrollterületek bevonásával a vasúti zöldfolyosók és barnamezők piaci reakciója több területen is számszerűsíthető, ami erősíti az eredmények külső érvényességét.
- **Várospolitikai tanulság**: A becsült immissziós diszkontok és az állomáskörzeti elérhetőségi prémiumok iránya és nagyságrendje alapján a zajvédelmi rehabilitáció és a gyalogos kapcsolatok fejlesztése a vizsgált területeken is értéknövekedést generálhat — a pontos hatások a fenti regressziós együtthatókból olvashatók le.
